# Data Integration Phase
Enriches data from silver lake and writes aggregated and enriched taxi_trips data as gold Delta table `integrated_taxi_trips`.


## 1. Configure Spark


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4a36c1f8-b1ac-40e7-ae6a-d400b50cb574;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 105ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs


## 2. Load silver tables

Read silver delta tables.

In [2]:
from pyspark.sql import functions as F

trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Hourly weather (NYC local)



In [3]:
hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("temperature_c").alias("temperature_c"),
        F.avg("wind_speed_ms").alias("wind_speed_ms"),
    )
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)


26/09/19 16:02:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


hourly_weather: 8,784 hours
+-----------+-----------+-------------+------------------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms     |
+-----------+-----------+-------------+------------------+
|2024-08-23 |2          |22.2         |3.1111111111111107|
|2024-08-23 |7          |20.6         |3.611111111111111 |
|2024-08-23 |11         |18.9         |0.0               |
+-----------+-----------+-------------+------------------+
only showing top 3 rows



## 4. Hourly air quality (NYC local)



In [4]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("value").alias("pm25"),
        F.first("unit").alias("pm25_unit"),
    )
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)


hourly_aq: 8,783 hours
+-----------+-----------+------------------+---------------------------+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |
+-----------+-----------+------------------+---------------------------+
|2024-01-01 |0          |14.520000000000001|Micrograms/cubic meter (LC)|
|2024-01-01 |1          |14.459999999999999|Micrograms/cubic meter (LC)|
|2024-01-01 |2          |14.440000000000001|Micrograms/cubic meter (LC)|
+-----------+-----------+------------------+---------------------------+
only showing top 3 rows



## 5. Pickup / dropoff zone lookups

`taxi_zones` is a small table ==> we can easily and with minimal overhead split them into `pickup_zones` and `dropoff_zones`


In [5]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)


## 6. Enrich trips and write `integrated_taxi_trips`

Left-join weather and air_quality data onto taxi_trip data.

In [6]:
integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .select(
        "taxi_type",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough",
        "dropoff_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "temperature_c",
        "wind_speed_ms",
        "pm25",
        "pm25_unit",
        "pickup_date",
        "pickup_hour",
    )
    .cache()
)

# Materialize the shared input once so both writes measure partitioning and I/O.
integrated.count()

import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
print(f"write by_date: {time.perf_counter() - t0:.1f}s")
show_delta(spark, GOLD / "integrated_taxi_trips")

write by_date: 11.8s
integrated_taxi_trips: 9417383 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-----------------------+--------------+-------------------+---------------+---------------+-----------+----------+------------+------------+-------------+-----------------+----+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone            |pickup_borough|dropoff_location_id|dropoff_zone   |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms    |pm25|pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-----------------------+--------------+-------------------+---------------+---------------+-----------+----------+

## 7. Two storage designs

To compare two storage designs, we can partition the same rows aggregated from silver to gold layer in two different ways:

| Partitioning Logic | Table | Partition | Suited for |
| --- | --- | --- | --- |
| By timestamp | `integrated_taxi_trips` | `pickup_date` | average duration per day |
| By location | `integrated_taxi_trips_by_borough` | `pickup_borough` | trip data per borough location |


In [7]:
import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")


def storage_report(table_name: str) -> None:
    path = GOLD / table_name
    files = [f for f in path.rglob("*.parquet") if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    n_parts = len({f.parent for f in files})
    print(
        f"{table_name:40} files={len(files):>5}  partitions={n_parts:>4}  size={size_mb:>8.1f} MB"
    )


print()
print("Storage")
storage_report("integrated_taxi_trips")
storage_report("integrated_taxi_trips_by_borough")

write by_borough: 8.4s

Storage
integrated_taxi_trips                    files=  222  partitions=  92  size=   227.7 MB
integrated_taxi_trips_by_borough         files=  200  partitions=   8  size=   226.9 MB


### Queries on both designs


In [8]:
import time


def queries(df):
    duration_min = (
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    ) / 60.0
    return {
        "trips per borough": (
            df.groupBy("pickup_borough")
            .agg(F.count(F.lit(1)).alias("trips"))
            .orderBy(F.desc("trips"))
        ),
        "avg duration per day": (
            df.withColumn("duration_min", duration_min)
            .groupBy("pickup_date")
            .agg(F.avg("duration_min").alias("avg_duration_min"))
            .orderBy("pickup_date")
        ),
        "avg fare per borough": (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("avg_fare"))
            .orderBy("pickup_borough")
        ),
    }


def run_queries(table_name: str) -> None:
    spark.catalog.clearCache()
    df = read_delta(spark, GOLD / table_name)
    print(f"\n{table_name}")
    print("-" * 40)
    for name, q in queries(df).items():
        t0 = time.perf_counter()
        rows = q.collect()
        elapsed = time.perf_counter() - t0
        print(f"\n{name}  ({elapsed:.2f}s, {len(rows):,} rows)")
        spark.createDataFrame(rows).show(20, truncate=False)


run_queries("integrated_taxi_trips")
run_queries("integrated_taxi_trips_by_borough")



integrated_taxi_trips
----------------------------------------

trips per borough  (0.54s, 8 rows)


+--------------+-------+
|pickup_borough|trips  |
+--------------+-------+
|Manhattan     |8442728|
|Queens        |817017 |
|Brooklyn      |95738  |
|Unknown       |31298  |
|Bronx         |24715  |
|N/A           |4752   |
|EWR           |915    |
|Staten Island |220    |
+--------------+-------+


avg duration per day  (0.74s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.444424507823307|
|2024-01-02 |16.906788016430834|
|2024-01-03 |16.545877362397537|
|2024-01-04 |16.08400669921135 |
|2024-01-05 |15.385487487153334|
|2024-01-06 |14.58333002828419 |
|2024-01-07 |13.901090108635216|
|2024-01-08 |15.62057394556443 |
|2024-01-09 |15.068740392765223|
|2024-01-10 |15.230732569936643|
|2024-01-11 |16.433418924474225|
|2024-01-12 |16.471563084834628|
|2024-01-13 |15.074412049124728|
|2024-01-14 |14.352769278119478|
|2024-01-15 |14.90966166434527 |
|2024-01-16 |16.602465760869592|
|2024-01-17 |16.343770248792744|


avg duration per day  (0.82s, 92 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2024-01-01 |16.44442450782354 |
|2024-01-02 |16.906788016430923|
|2024-01-03 |16.54587736239775 |
|2024-01-04 |16.08400669921109 |
|2024-01-05 |15.38548748715337 |
|2024-01-06 |14.583330028284262|
|2024-01-07 |13.901090108635143|
|2024-01-08 |15.620573945564523|
|2024-01-09 |15.068740392765537|
|2024-01-10 |15.23073256993671 |
|2024-01-11 |16.433418924474434|
|2024-01-12 |16.47156308483458 |
|2024-01-13 |15.074412049124797|
|2024-01-14 |14.352769278119418|
|2024-01-15 |14.90966166434513 |
|2024-01-16 |16.6024657608695  |
|2024-01-17 |16.34377024879265 |
|2024-01-18 |16.058340590670788|
|2024-01-19 |15.139623995861927|
|2024-01-20 |14.266279737782732|
+-----------+------------------+
only showing top 20 rows


avg fare per borough  (0.65s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+---------------

### Query 1

In [9]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_1 = """
WITH monthly_zone_trips AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        pickup_location_id,
        pickup_borough,
        pickup_zone,
        pickup_date
    FROM integrated_taxi_trips
    WHERE pickup_zone != 'UNKNOWN'
      AND pickup_date IS NOT NULL
)
SELECT
    trip_month,
    pickup_location_id,
    pickup_borough,
    pickup_zone,
    COUNT(*) AS total_trips,
    COUNT(DISTINCT pickup_date) AS active_days,
    ROUND(
        CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
        2
    ) AS avg_daily_trips
FROM monthly_zone_trips
GROUP BY trip_month, pickup_location_id, pickup_borough, pickup_zone
ORDER BY trip_month ASC, total_trips DESC
"""

monthly_demand = spark.sql(query_1)
monthly_demand.show(20, truncate=False)

+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|trip_month|pickup_location_id|pickup_borough|pickup_zone                 |total_trips|active_days|avg_daily_trips|
+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|2024-01-01|161               |Manhattan     |Midtown Center              |141738     |31         |4572.19        |
|2024-01-01|237               |Manhattan     |Upper East Side South       |141263     |31         |4556.87        |
|2024-01-01|132               |Queens        |JFK Airport                 |141159     |31         |4553.52        |
|2024-01-01|236               |Manhattan     |Upper East Side North       |135334     |31         |4365.61        |
|2024-01-01|162               |Manhattan     |Midtown East                |105466     |31         |3402.13        |
|2024-01-01|230               |Manhattan     |Times Sq/Theatre District 

### Query 2

In [10]:
query_2 = """
WITH binned_weather AS (
    SELECT
        trip_distance,
        CASE
            WHEN temperature_c IS NULL THEN 'Unknown'
            WHEN temperature_c < 0 THEN 'Freezing (<0°C)'
            WHEN temperature_c BETWEEN 0 AND 10 THEN 'Cold (0°C to 10°C)'
            WHEN temperature_c BETWEEN 10.01 AND 20 THEN 'Moderate (10°C to 20°C)'
            ELSE 'Warm (>20°C)'
        END AS temp_category,
        CASE
            WHEN wind_speed_ms IS NULL THEN 'Unknown'
            WHEN wind_speed_ms < 2 THEN 'Calm (<2 m/s)'
            WHEN wind_speed_ms BETWEEN 2 AND 6 THEN 'Moderate Wind (2-6 m/s)'
            ELSE 'High Wind (>6 m/s)'
        END AS wind_category
    FROM integrated_taxi_trips
    WHERE trip_distance > 0 AND trip_distance < 100
)
SELECT
    temp_category,
    wind_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles
FROM binned_weather
GROUP BY temp_category, wind_category
ORDER BY temp_category, wind_category
"""

avg_distance_weather = spark.sql(query_2)
avg_distance_weather.show(20, truncate=False)

+-----------------------+-----------------------+----------+------------------+
|temp_category          |wind_category          |trip_count|avg_distance_miles|
+-----------------------+-----------------------+----------+------------------+
|Cold (0°C to 10°C)     |Calm (<2 m/s)          |395811    |3.29              |
|Cold (0°C to 10°C)     |High Wind (>6 m/s)     |2272210   |3.31              |
|Cold (0°C to 10°C)     |Moderate Wind (2-6 m/s)|4128105   |3.29              |
|Freezing (<0°C)        |Calm (<2 m/s)          |13866     |4.2               |
|Freezing (<0°C)        |High Wind (>6 m/s)     |478247    |3.23              |
|Freezing (<0°C)        |Moderate Wind (2-6 m/s)|622618    |3.2               |
|Moderate (10°C to 20°C)|Calm (<2 m/s)          |117228    |3.37              |
|Moderate (10°C to 20°C)|High Wind (>6 m/s)     |460189    |3.28              |
|Moderate (10°C to 20°C)|Moderate Wind (2-6 m/s)|669868    |3.41              |
|Warm (>20°C)           |Calm (<2 m/s)  

### Query 3

In [11]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_3 = """
WITH rounded_pm25 AS (
    SELECT
        ROUND(pm25, 0) AS pm25_level,
        pickup_date,
        pickup_hour
    FROM integrated_taxi_trips
    WHERE pm25 IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
)
SELECT
    pm25_level,
    COUNT(*) AS trips,
    COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
    ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT struct(pickup_date, pickup_hour)), 2) AS trips_per_hour
FROM rounded_pm25
GROUP BY pm25_level
ORDER BY pm25_level DESC
"""

pm25_demand = spark.sql(query_3)
pm25_demand.show(34, truncate=False)

+----------+-------+--------------+--------------+
|pm25_level|trips  |observed_hours|trips_per_hour|
+----------+-------+--------------+--------------+
|34.0      |4188   |1             |4188.0        |
|33.0      |9071   |2             |4535.5        |
|31.0      |5019   |1             |5019.0        |
|30.0      |18438  |3             |6146.0        |
|29.0      |18240  |2             |9120.0        |
|28.0      |17702  |3             |5900.67       |
|27.0      |14835  |2             |7417.5        |
|26.0      |66047  |10            |6604.7        |
|25.0      |18435  |3             |6145.0        |
|24.0      |54765  |10            |5476.5        |
|23.0      |20069  |5             |4013.8        |
|22.0      |54775  |12            |4564.58       |
|21.0      |80355  |17            |4726.76       |
|20.0      |83362  |16            |5210.13       |
|19.0      |79234  |17            |4660.82       |
|18.0      |132230 |31            |4265.48       |
|17.0      |105092 |29         

### Query 4

In [12]:
query = """
WITH trips_with_weather AS (
    SELECT 
        pickup_zone,
        pickup_date,
        pickup_hour,
        NTILE(4) OVER (
            PARTITION BY pickup_zone
            ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)
        ) AS weather_condition
    FROM integrated_taxi_trips
    WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
),
hourly_demand AS (
    SELECT 
        pickup_zone,
        weather_condition,
        COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
    FROM trips_with_weather
    GROUP BY pickup_zone, weather_condition
),
pivoted AS (
    SELECT * FROM hourly_demand
    PIVOT (
        ROUND(AVG(trips_per_hour), 2)
        FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
    )
)
SELECT 
    pickup_zone,
    coldest, cool, warm, warmest,
    ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
FROM pivoted
WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
ORDER BY pct_variation DESC
"""

spark.sql(query).show(truncate=False)

+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.25  |13.0  |12.66 |10.69  |42.28        |
|Financial District North     |26.62  |21.65 |21.07 |18.24  |38.27        |
|Meatpacking/West Village West|49.81  |38.38 |40.34 |34.71  |37.0         |
|Lower East Side              |58.06  |45.25 |48.38 |41.47  |34.35        |
|Little Italy/NoLiTa          |52.69  |41.92 |43.93 |38.29  |32.57        |
|Greenwich Village South      |75.86  |62.62 |63.19 |57.27  |28.72        |
|West Village                 |121.51 |101.08|101.63|92.03  |28.33        |
|Battery Park City            |31.51  |28.26 |26.96 |23.97  |27.24        |
|TriBeCa/Civic Center         |66.51  |58.87 |57.36 |50.75  |27.0         |
|World Trade Center           |25.62  |21.19 |21.69 |19.84  |26.17        |
|East Villag

### Query 5 — Peak travel hours for each day of the week

In [13]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_5 = """
WITH hourly_by_dow AS (
    SELECT
        CASE dayofweek(pickup_date)
            WHEN 1 THEN 'Sunday'
            WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'
            WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday'
            WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
        END AS day_of_week,
        CASE dayofweek(pickup_date)
            WHEN 1 THEN 7
            ELSE dayofweek(pickup_date) - 1
        END AS dow_order,
        pickup_hour,
        COUNT(*) AS trip_count,
        COUNT(DISTINCT pickup_date) AS active_days,
        ROUND(
            CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
            2
        ) AS avg_trips
    FROM integrated_taxi_trips
    WHERE pickup_date IS NOT NULL
      AND pickup_hour IS NOT NULL
    GROUP BY dayofweek(pickup_date), pickup_hour
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY day_of_week
            ORDER BY avg_trips DESC, trip_count DESC
        ) AS peak_rank
    FROM hourly_by_dow
)
SELECT
    day_of_week,
    pickup_hour AS peak_hour,
    trip_count,
    active_days,
    avg_trips
FROM ranked
WHERE peak_rank = 1
ORDER BY dow_order
"""

peak_hours = spark.sql(query_5)
peak_hours.show(truncate=False)

+-----------+---------+----------+-----------+---------+
|day_of_week|peak_hour|trip_count|active_days|avg_trips|
+-----------+---------+----------+-----------+---------+
|Monday     |18       |80482     |13         |6190.92  |
|Tuesday    |18       |99614     |13         |7662.62  |
|Wednesday  |18       |110914    |13         |8531.85  |
|Thursday   |18       |119124    |13         |9163.38  |
|Friday     |18       |102921    |13         |7917.0   |
|Saturday   |19       |96485     |13         |7421.92  |
|Sunday     |0        |79601     |13         |6123.15  |
+-----------+---------+----------+-----------+---------+



### Query 6 — Monthly trends in taxi demand

In [14]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_6 = """
WITH monthly AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        COUNT(*) AS total_trips,
        COUNT(DISTINCT pickup_date) AS active_days,
        ROUND(
            CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
            2
        ) AS avg_daily_trips
    FROM integrated_taxi_trips
    WHERE pickup_date IS NOT NULL
    GROUP BY TRUNC(pickup_date, 'MM')
)
SELECT
    trip_month,
    total_trips,
    active_days,
    avg_daily_trips,
    LAG(avg_daily_trips) OVER (ORDER BY trip_month) AS prev_month_avg_daily,
    ROUND(
        100.0 * (
            avg_daily_trips - LAG(avg_daily_trips) OVER (ORDER BY trip_month)
        ) / LAG(avg_daily_trips) OVER (ORDER BY trip_month),
        2
    ) AS pct_change_vs_prev_month
FROM monthly
ORDER BY trip_month
"""

monthly_trends = spark.sql(query_6)
monthly_trends.show(truncate=False)

26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+-----------+---------------+--------------------+------------------------+
|trip_month|total_trips|active_days|avg_daily_trips|prev_month_avg_daily|pct_change_vs_prev_month|
+----------+-----------+-----------+---------------+--------------------+------------------------+
|2024-01-01|2926910    |31         |94416.45       |NULL                |NULL                    |
|2024-02-01|2966705    |29         |102300.17      |94416.45            |8.35                    |
|2024-03-01|3523766    |31         |113669.87      |102300.17           |11.11                   |
|2024-04-01|2          |1          |2.0            |113669.87           |-100.0                  |
+----------+-----------+-----------+---------------+--------------------+------------------------+



26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/19 16:19:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
